# Install Python packages and download dataset

# Import Python Modules

In [1]:
from data_process import prepare, read_image

import os
from PIL import Image
from mat73 import loadmat

import tensorflow as tf
import tensorflow.keras as keras
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
# to perform subject-wise cross validation
from sklearn.model_selection import GroupKFold, train_test_split
import cv2

2025-03-23 12:03:50.048305: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-03-23 12:03:50.059431: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-03-23 12:03:50.062987: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-03-23 12:03:50.072490: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-03-23 12:03:50.638114: W tensorflow/compiler/tf2

In [2]:
tf.config.list_physical_devices('GPU')

I0000 00:00:1742745831.835876 2169739 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1742745831.872964 2169739 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1742745831.875822 2169739 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355


[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

## Load images, masks and labels

In [3]:
labels = np.load('./dataset/labels.npy')
images = np.load('./dataset/images.npy', mmap_mode='r')
masks = np.load('./dataset/masks.npy', mmap_mode='r')
patient_ids = np.load('./dataset/patient_ids.npy').flatten()
file_paths = np.load('./dataset/file_paths.npy')

integer_to_class = {'1': 'meningioma (1)', '2': 'glioma (2)', '3': 'pituitary tumor (3)'}

# Create dataset splits




## One test

To test our implementation quickly we used the following split, without considering patient IDs: 60% of examples are placed in the training dataset, 20% are placed in the validation dataset, and the remaining 20% are used for the test set.

In [4]:
X, X_test, y, y_test = train_test_split(file_paths, labels, test_size=0.2, random_state=1)
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.25)


train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train))
val_ds = tf.data.Dataset.from_tensor_slices((X_val, y_val))
test_ds = tf.data.Dataset.from_tensor_slices((X_test, y_test))

I0000 00:00:1742745835.291619 2169739 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1742745835.294726 2169739 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1742745835.297339 2169739 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1742745835.430596 2169739 cuda_executor.cc:1015] successful NUMA node read from SysFS ha

## Report on balance of classes in each of the datasets

In [5]:

def summarize_dataset(ds, name):
    num_classes = 3
    count = np.zeros(num_classes, dtype=np.int32)
    for _, label in ds:
      count[label.numpy() - 1] += 1

    total = np.sum(count)

    label_stats = ""

    for (id_, label) in integer_to_class.items():
      label_stats += "{} {}\n".format(integer_to_class[id_], count[int(id_) - 1])

    print(f"Distribution of labels for {name}")
    print(label_stats)
    return

if train_ds is not None:
  summarize_dataset(train_ds, "Training Dataset")
  summarize_dataset(val_ds, "Validation Dataset")
  summarize_dataset(test_ds, "Test Dataset")

Distribution of labels for Training Dataset
meningioma (1) 407
glioma (2) 858
pituitary tumor (3) 573

Distribution of labels for Validation Dataset
meningioma (1) 153
glioma (2) 282
pituitary tumor (3) 178

Distribution of labels for Test Dataset
meningioma (1) 148
glioma (2) 286
pituitary tumor (3) 179



2025-03-23 12:03:58.302535: I tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
2025-03-23 12:03:58.368136: I tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


## Subject-Wise Cross Validation


With subject-wise cross validation, we ensure that a patient's scans do not appear in both the training and test sets. We split the dataset into **folds**,
where each fold has roughly the same number of unique patients. The function
`assign_folds_custom` picks six folds for training, 2 for validation, and 2 for testing. `assign_folds_custom` is designed to be called in a loop, to return a new (train_set, valid_set, test_set) combination on each iteration.

In [6]:
# Each patient is limited to appearing in one fold. The number of distinct
# patients in each fold is approximately the same
group_kfold = GroupKFold(n_splits=10, shuffle=True)

# A list of lists, where each sublist represents the indices corresponding to a
# single fold
folds = [indices for (_, indices) in group_kfold.split(file_paths, labels, patient_ids)]
# first two folds to be held out as a test set
test_indices = np.concatenate((folds[0], folds[1])) 
folds = folds[2:]

# Take 1 fold for validation, 7 for training
def assign_folds_custom():

  train_indices = np.array([], dtype=np.int16)
  val_indices = np.array([], dtype=np.int16)

  for i in range(8):
      j = i
      for _ in range(7):
        train_indices = np.append(train_indices, folds[j])
        j = (j + 1) % len(folds)
    
      val_indices = np.append(val_indices, folds[j])
      train_ds = tf.data.Dataset.from_tensor_slices((file_paths[train_indices], labels[train_indices]))
      val_ds = tf.data.Dataset.from_tensor_slices((file_paths[val_indices], labels[val_indices]))

      yield train_ds, val_ds

## Data Preprocessing

* Resize every image to 256 $\times$ 256 pixels.
* Load the dataset in batches of 32 images so that batches can be stored on GPU memory.
* Scalar-valued targets should be converted to one-hot vectors.

In [7]:
train_ds = prepare(train_ds, shuffle=True, augment=True)
val_ds = prepare(val_ds, shuffle=True, augment=False)
test_ds = prepare(test_ds, shuffle=True, augment=False)

In [9]:
resize = keras.Sequential([
keras.layers.Resizing(256, 256)
])


normalize = keras.layers.Normalization(axis=(1, 2))
adapt_data = train_ds.map(lambda x, y: resize(x))
normalize.adapt(adapt_data)

# Model Implementation

In [10]:
model = keras.Sequential()
model.name = "classifier_to_explain"

### Pre-Processing Layer ###
model.add(keras.Input(shape=(512, 512, 1)))
model.add(keras.layers.Resizing(256, 256))
model.add(normalize)

### Classification Block A ###

# Layers 2 and 3
model.add(keras.layers.Conv2D(16, kernel_size=(5, 5), strides=(2, 2), padding='same', activation='relu'))

# Layer 4 - Dropout
model.add(keras.layers.Dropout(0.5))

# Layer 5 - Max Pooling
model.add(keras.layers.MaxPooling2D(pool_size=(2, 2), strides=(2, 2), padding='valid'))

# Modification - add batch norm layer
model.add(keras.layers.BatchNormalization())


# Layer 6, 7 - Convolutional, ReLU
model.add(keras.layers.Conv2D(32, kernel_size=(3, 3), strides=(2, 2), padding='same', activation='relu'))

# Layer 8 - Dropout
model.add(keras.layers.Dropout(0.5))

# Layer 9 - 2D Max Pooling
model.add(keras.layers.MaxPooling2D(pool_size=(2, 2), strides=(2, 2), padding='valid'))
model.add(keras.layers.BatchNormalization())

# Layer 10, 11 - Convolutional, ReLU
model.add(keras.layers.Conv2D(64, kernel_size=(3, 3), strides=(1, 1), padding='same', activation='relu'))

# Layer 12 - Dropout
model.add(keras.layers.Dropout(0.5))

# Layer 13 - 2D Max Pooling
model.add(keras.layers.MaxPooling2D(pool_size=(2, 2), strides=(2, 2), padding='valid'))

model.add(keras.layers.BatchNormalization())

# Layer 14, 15 - Convolutional, ReLU
model.add(keras.layers.Conv2D(128, kernel_size=(3, 3), strides=(1, 1), padding='same', activation='relu'))

# Layer 16 - Dropout
model.add(keras.layers.Dropout(0.5))

# Layer 17 - Max Pooling
model.add(keras.layers.MaxPooling2D(pool_size=(2, 2), strides=(2, 2), padding='valid'))

# Flattening output
model.add(keras.layers.Flatten())

# Layer 18, 19 - Fully Connected, ReLU
model.add(keras.layers.Dense(2048, activation='relu'))
model.add(keras.layers.Dropout(0.5))

# Layer 20, 21 - Fully Connected, Softmax
model.add(keras.layers.Dense(units=3, activation='softmax'))

model.summary()

Model: "classifier_to_explain"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ resizing_2 (Resizing)           │ (None, 256, 256, 1)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ normalization_1 (Normalization) │ (None, 256, 256, 1)    │       131,073 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 128, 128, 16)   │           416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128, 128, 16)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 64, 64, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 64, 64, 16)     │            64 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 32, 32, 32)     │         4,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32, 32, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 16, 16, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 16, 16, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 16, 16, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 16, 16, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 8, 8, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 8, 8, 64)       │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 8, 8, 128)      │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 8, 8, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 4, 4, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 2048)           │     4,196,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 3)              │         6,147 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,431,428 (16.90 MB)

 Trainable params: 4,300,131 (16.40 MB)

 Non-trainable params: 131,297 (512.88 KB)

In [23]:
# Define a callback to save the model with the lowest validation loss
def train_once(train_ds, val_ds, test_ds, epochs=100, trial_id=0):

  output_directory = os.getcwd() + "/saved_models"
  model_filename = f"best_model-trial-{trial_id}.keras"
  model_path = os.path.join(output_directory, model_filename)

  model_checkpoint = keras.callbacks.ModelCheckpoint(model_path,
                                                    save_best_only=True,
                                                    save_weights_only=False,
                                                    monitor="val_loss",
                                                    mode="min", verbose=1)


  # Adapt the normalization layer of the model
  model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001), loss='categorical_crossentropy', metrics=['accuracy', 'auc'])

  # Define a callback for early stopping
  early_stopping = keras.callbacks.EarlyStopping(monitor="val_loss", patience=11,
                                                mode="min", verbose=1 if epochs % 10 == 0 else 0)

  # Training configuration
  epochs = 100  # You can adjust the number of epochs

  # Train the model with callbacks
  history = model.fit(
      x=train_ds,  # Provide the input data directly from the dataset
      epochs=epochs,
      validation_data=val_ds,
      callbacks=[model_checkpoint, early_stopping]
  )

  return history



## Training: One-Test

In [13]:
history = train_once(train_ds, val_ds, test_ds)
print(history)

Epoch 1/100


I0000 00:00:1742745884.232809 2169859 service.cc:146] XLA service 0x71e6b0005eb0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1742745884.232828 2169859 service.cc:154]   StreamExecutor device (0): NVIDIA GeForce RTX 3090, Compute Capability 8.6
2025-03-23 12:04:44.314547: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2025-03-23 12:04:44.652614: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:531] Loaded cuDNN version 8907


 24/173 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.4720 - auc: 0.6260 - loss: 6.3198

I0000 00:00:1742745888.428884 2169859 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


173/173 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step - accuracy: 0.6056 - auc: 0.7647 - loss: 2.5043

2025-03-23 12:04:59.135512: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:393] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_240', 236 bytes spill stores, 236 bytes spill loads

2025-03-23 12:05:00.250780: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:393] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_240', 12 bytes spill stores, 12 bytes spill loads




Epoch 1: val_loss improved from inf to 1.31391, saving model to /home/worathur/Documents/CV-Capstone/saved_models/best_model.keras
173/173 ━━━━━━━━━━━━━━━━━━━━ 19s 73ms/step - accuracy: 0.6060 - auc: 0.7651 - loss: 2.4969 - val_accuracy: 0.2496 - val_auc: 0.4688 - val_loss: 1.3139
Epoch 2/100
170/173 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step - accuracy: 0.7107 - auc: 0.8904 - loss: 0.6278 

KeyboardInterrupt: 

In [ ]:
best_train_acc = max(history.history['accuracy'])
best_valid_acc = max(history.history['val_accuracy'])


print(f"Best Training Accuracy: {best_train_acc}")
print(f"Best Validation Accuracy: {best_valid_acc}")

metrics = model.evaluate(x=test_ds, return_dict=True)
test_accuracy = metrics['accuracy']

print(f"Final Test Accuracy: {test_accuracy}")

In [ ]:
# # Define needed variables
tr_acc = history.history['accuracy']
tr_loss = history.history['loss']
tr_auroc = histor.history['auc']
val_acc = history.history['val_accuracy']
val_loss = history.history['val_loss']
val_auroc = history.history['val_auc']
index_loss = np.argmin(val_loss)
val_lowest = val_loss[index_loss]
index_acc = np.argmax(val_acc)
acc_highest = val_auroc[index_acc]
Epochs = [i+1 for i in range(len(tr_acc))]
loss_label = f'best epoch= {str(index_loss + 1)}'
acc_label = f'best epoch= {str(index_acc + 1)}'

# Plot training history
plt.figure(figsize= (20, 8))
plt.style.use('fivethirtyeight')

plt.subplot(1, 2, 1)
plt.plot(Epochs, tr_loss, 'r', label= 'Training loss')
plt.plot(Epochs, val_loss, 'g', label= 'Validation loss')
plt.scatter(index_loss + 1, val_lowest, s= 150, c= 'blue', label= loss_label)
plt.title('Training and Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(Epochs, tr_auc, 'r', label= 'Training Accuracy')
plt.plot(Epochs, val_auc, 'g', label= 'Validation Accuracy')
plt.scatter(index_acc + 1 , acc_highest, s= 150, c= 'blue', label= acc_label)
plt.title('Training and Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()

plt.tight_layout
plt.show()

## Training: Subject-Wise 10-Fold Cross Validation

In [24]:
labels = np.load('./dataset/labels.npy')
images = np.load('./dataset/images.npy', mmap_mode='r')
masks = np.load('./dataset/masks.npy', mmap_mode='r')
patient_ids = np.load('./dataset/patient_ids.npy').flatten()
file_paths = np.load('./dataset/file_paths.npy')

integer_to_class = {'1': 'meningioma (1)', '2': 'glioma (2)', '3': 'pituitary tumor (3)'}

In [ ]:

test_ds = tf.data.Dataset.from_tensor_slices((file_paths[test_indices], labels[test_indices]))
test_ds = prepare(test_ds, shuffle=True, augment=False)

avg_accuracy = 0
for i, (train_ds, val_ds) in enumerate(assign_folds_custom()):
  print(f"=== Trial {i + 1} ===")

  train_ds = prepare(train_ds, shuffle=True, augment=True)
  val_ds = prepare(val_ds, shuffle=True, augment=False)
  

  history = train_once(train_ds, val_ds, test_ds, epochs=35, trial_id=i)

  best_train_acc = max(history.history['accuracy'])
  best_valid_acc = max(history.history['val_accuracy'])


  print(f"Best Training Accuracy: {best_train_acc}")
  print(f"Best Validation Accuracy: {best_valid_acc}")

  metrics = model.evaluate(x=test_ds, return_dict=True)
  test_accuracy = metrics['accuracy']

  print(f"Final Test Accuracy: {test_accuracy}")
  avg_accuracy += test_accuracy

print(f"Average Test Accuracy: {avg_accuracy}")

=== Trial 1 ===
Epoch 1/100

Epoch 1: val_loss improved from inf to 2.13842, saving model to /home/worathur/Documents/CV-Capstone/saved_models/best_model-trial-0.keras
196/196 ━━━━━━━━━━━━━━━━━━━━ 24s 91ms/step - accuracy: 0.9144 - auc: 0.9840 - loss: 0.2410 - val_accuracy: 0.1955 - val_auc: 0.4476 - val_loss: 2.1384
Epoch 2/100

Epoch 2: val_loss improved from 2.13842 to 1.42904, saving model to /home/worathur/Documents/CV-Capstone/saved_models/best_model-trial-0.keras
196/196 ━━━━━━━━━━━━━━━━━━━━ 17s 81ms/step - accuracy: 0.9024 - auc: 0.9838 - loss: 0.2405 - val_accuracy: 0.1955 - val_auc: 0.4573 - val_loss: 1.4290
Epoch 3/100

Epoch 3: val_loss improved from 1.42904 to 1.15274, saving model to /home/worathur/Documents/CV-Capstone/saved_models/best_model-trial-0.keras
196/196 ━━━━━━━━━━━━━━━━━━━━ 17s 80ms/step - accuracy: 0.9031 - auc: 0.9834 - loss: 0.2474 - val_accuracy: 0.5042 - val_auc: 0.6020 - val_loss: 1.1527
Epoch 4/100

Epoch 4: val_loss did not improve from 1.15274
196/196